# Extract secondary-type flags

Builds a per-album lookup marking which release groups are **Live** or **Compilation**, so the app's content-filter faders (Live Albums / Greatest Hits) can work.

MusicBrainz `release_group_secondary_type`:
- `1` = Compilation
- `6` = Live

Output: `data/raw/mb_album_secondary_type.parquet` with columns `album_id, is_live, is_compilation`.

In [ ]:
import os
import duckdb
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
PG = (f"host={os.getenv('PG_HOST','localhost')} port={os.getenv('PG_PORT','5432')} "
      f"dbname={os.getenv('PG_DBNAME','musicbrainz_db')} "
      f"user={os.getenv('PG_USER','musicbrainz')} password={os.getenv('PG_PASSWORD','musicbrainz')}")

con = duckdb.connect()
con.execute("INSTALL postgres; LOAD postgres;")
con.execute(f"ATTACH '{PG}' AS mb (TYPE postgres, READ_ONLY);")

# release_group.id  ←  release_group_secondary_type_join.release_group
#                       release_group_secondary_type_join.secondary_type → 1=Comp, 6=Live
df = con.execute("""
    SELECT
        rg.id AS album_id,
        MAX(CASE WHEN j.secondary_type = 6 THEN 1 ELSE 0 END) AS is_live,
        MAX(CASE WHEN j.secondary_type = 1 THEN 1 ELSE 0 END) AS is_compilation
    FROM mb.musicbrainz.release_group rg
    LEFT JOIN mb.musicbrainz.release_group_secondary_type_join j
           ON j.release_group = rg.id
    GROUP BY rg.id
""").fetch_df()

con.close()

df['is_live'] = df['is_live'].astype('int8')
df['is_compilation'] = df['is_compilation'].astype('int8')
print(f'{len(df):,} release groups')
print(f"  live:        {df['is_live'].sum():,} ({df['is_live'].mean():.1%})")
print(f"  compilation: {df['is_compilation'].sum():,} ({df['is_compilation'].mean():.1%})")
df.head()

In [ ]:
os.makedirs('./data/raw', exist_ok=True)
out = './data/raw/mb_album_secondary_type.parquet'
df.to_parquet(out, compression='zstd')
print(f'Saved {out}  ({os.path.getsize(out)/1024**2:.1f} MB)')